# Depth Completion Prototype — CompletionFormer on nuScenes

Validates the O4 pipeline step: sparse aggregated LiDAR → CompletionFormer → dense metric depth → (H,W,3) pointmap.

**Scope**: no SAM3, no mask filtering. Global pipeline only.

**Steps**:
1. Load nuScenes frame + aggregated pre-filtered LiDAR (existing pipeline)
2. Project to sparse `(H,W)` depth map — min depth per pixel
3. Run CompletionFormer → dense metric depth
4. Back-project → `(H,W,3)` pointmap in PyTorch3D convention
5. Visualise: image | sparse input | completed depth | point cloud overlay

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib
import matplotlib.pyplot as plt
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'   # 'nuscenes_mini' or 'ecp'

if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/workspace/data/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3
    FRAME_IDX    = 20    # pick a keyframe for meaningful GT comparison
    CAMERA       = 'CAM_FRONT'
    N_BEFORE, N_AFTER = 3, 3   # sweep aggregation; set 0,0 for single sweep
    LIDAR_LINES  = 32          # nuScenes: 32-beam Velodyne
elif DATASET == 'ecp':
    DATA_ROOT    = '/workspace/data/ecp'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10
    FRAME_IDX    = 560
    CAMERA       = 'CAM_FRONT'
    N_BEFORE, N_AFTER = 2, 2
    LIDAR_LINES  = 64          # ECP: 64-beam LiDAR
else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}')

DEV_ROOT    = Path('/workspace')
CFORMER_ROOT = DEV_ROOT / 'models' / 'CompletionFormer'
CFORMER_CKPT = CFORMER_ROOT / 'checkpoints' / 'KITTIDC' / 'KITTIDC_L1L2.pt'

# LiDAR pre-filters (same values as full pipeline)
USE_EGO_BODY_FILTER = True
EGO_BOX_HALF_X = 4.0
EGO_BOX_HALF_Y = 1.5
EGO_BOX_Z_MIN  = 0.5
EGO_BOX_Z_MAX  = 2.5
MAX_LIDAR_RANGE_M = 52.0

print(f'Dataset : {DATASET}  scene={SCENE_IDX}  frame={FRAME_IDX}  camera={CAMERA}')
print(f'LiDAR   : {LIDAR_LINES}-beam')
print(f'Aggregation: N_BEFORE={N_BEFORE}  N_AFTER={N_AFTER}  (set both to 0 for single sweep)')
print(f'Checkpoint : {CFORMER_CKPT}')
print(f'Exists     : {CFORMER_CKPT.exists()}')


In [ ]:
# Add autolabeling src so we can reuse existing LiDAR loaders and filters
AL_SRC = str(DEV_ROOT / 'autolabeling' / 'src')
if AL_SRC not in sys.path:
    sys.path.insert(0, AL_SRC)

from autolabeling.data.nuscenes_loader import FrameRecord
from autolabeling.utils.lidar import load_lidar_pts, load_lidar_pts_aggregated, filter_lidar_pts
print('Autolabeling utils loaded.')


In [ ]:
nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)

scene   = nusc.scene[SCENE_IDX]
token   = nusc.get('sample', scene['first_sample_token'])['data'][CAMERA]
for _ in range(FRAME_IDX):
    rec   = nusc.get('sample_data', token)
    token = rec['next']
    assert token, f'FRAME_IDX {FRAME_IDX} exceeds scene length'

rec      = nusc.get('sample_data', token)
img_path = nusc.get_sample_data_path(token)
cal      = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])

K     = np.array(cal['camera_intrinsic'], dtype=np.float64)
R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
t_c2e = np.array(cal['translation'], dtype=np.float64)

img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W    = img_rgb.shape[:2]
cam_sd_token = token

ep      = nusc.get('ego_pose', rec['ego_pose_token'])
R_e2g   = Quaternion(ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g   = np.array(ep['translation'], dtype=np.float64)

print(f'Image: {W}x{H}  path: {img_path}')
print(f'K[0,0]={K[0,0]:.1f}  K[1,1]={K[1,1]:.1f}  cx={K[0,2]:.1f}  cy={K[1,2]:.1f}')
plt.figure(figsize=(14,4)); plt.imshow(img_rgb); plt.axis('off')
plt.title(f'Scene {SCENE_IDX} ({scene["name"]}), frame {FRAME_IDX} — {CAMERA}'); plt.show()


In [ ]:
# ── Inline LiDAR loading — mirrors lidar_filtering_exploration §4 exactly ─────
# Key fix: uses is_key_frame check to find the CLOSEST LiDAR sweep to the
# camera timestamp (not always the keyframe LIDAR_TOP), and derives anchor ego
# pose from the camera sample_data directly.  Per-sweep printout matches so
# you can compare line-by-line with lidar_filtering_exploration.

from pyquaternion import Quaternion as _Q

def _dc_walk_lidar_tokens(nusc, anchor_token, n_before, n_after):
    anchor_path = nusc.get_sample_data_path(anchor_token)
    seen_paths  = {anchor_path}
    backward = []
    tok = nusc.get('sample_data', anchor_token)['prev']
    while tok and len(backward) < n_before:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            backward.append(tok)
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['prev']
    tokens = []
    for i, t in enumerate(reversed(backward)):
        tokens.append((-(len(backward) - i), t))
    tokens.append((0, anchor_token))
    count = 0
    tok = nusc.get('sample_data', anchor_token)['next']
    while tok and count < n_after:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            count += 1
            tokens.append((count, tok))
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['next']
    return tokens

def _dc_load_sweep(nusc, tok, R_e2g_anchor, t_e2g_anchor):
    """Load one LiDAR sweep → anchor ego frame (N, 3) float64."""
    sd       = nusc.get('sample_data', tok)
    lid_cal  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    ego_pose = nusc.get('ego_pose', sd['ego_pose_token'])
    lid_path = nusc.get_sample_data_path(tok)
    R_l2e    = _Q(lid_cal['rotation']).rotation_matrix.astype(np.float64)
    t_l2e    = np.array(lid_cal['translation'], dtype=np.float64)
    R_e2g_sw = _Q(ego_pose['rotation']).rotation_matrix.astype(np.float64)
    t_e2g_sw = np.array(ego_pose['translation'], dtype=np.float64)
    pts_raw  = np.fromfile(lid_path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)
    pts_ego_sw  = (R_l2e @ pts_raw.T).T + t_l2e
    # ── Ego-body filter in sweep's own ego frame (vehicle always at origin) ──
    if USE_EGO_BODY_FILTER:
        _sw_box = (
            (np.abs(pts_ego_sw[:, 0]) < EGO_BOX_HALF_X) &
            (np.abs(pts_ego_sw[:, 1]) < EGO_BOX_HALF_Y) &
            (pts_ego_sw[:, 2] > EGO_BOX_Z_MIN) &
            (pts_ego_sw[:, 2] < EGO_BOX_Z_MAX)
        )
        pts_ego_sw = pts_ego_sw[~_sw_box]
    pts_global  = (R_e2g_sw @ pts_ego_sw.T).T + t_e2g_sw
    pts_ego_anc = (R_e2g_anchor.T @ (pts_global - t_e2g_anchor).T).T
    return pts_ego_anc, Path(lid_path).name

# ── Anchor ego pose from camera sample_data (camera timestamp) ────────────────
cam_sd  = nusc.get('sample_data', cam_sd_token)
cam_ts  = cam_sd['timestamp']
cam_ep  = nusc.get('ego_pose', cam_sd['ego_pose_token'])
R_e2g_anchor = _Q(cam_ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g_anchor = np.array(cam_ep['translation'], dtype=np.float64)

# ── Find anchor LiDAR sweep — closest to camera timestamp ────────────────────
# Keyframe: use sample's LIDAR_TOP directly (same as lidar_filtering_exploration).
# Non-keyframe: search full LiDAR SD chain for the closest sweep.
if cam_sd['is_key_frame']:
    _sample     = nusc.get('sample', cam_sd['sample_token'])
    lidar_token = _sample['data']['LIDAR_TOP']
else:
    lidar_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - cam_ts),
    )['token']

# ── Load & aggregate ──────────────────────────────────────────────────────────
if N_BEFORE > 0 or N_AFTER > 0:
    sweep_tokens = _dc_walk_lidar_tokens(nusc, lidar_token, N_BEFORE, N_AFTER)
    all_pts = []
    print(f'LiDAR aggregation: {len(sweep_tokens)} unique sweep(s)')
    for rel_idx, tok in sweep_tokens:
        sweep_pts, fname = _dc_load_sweep(nusc, tok, R_e2g_anchor, t_e2g_anchor)
        sd_info = nusc.get('sample_data', tok)
        dt_ms   = (sd_info['timestamp'] - cam_ts) / 1_000.0
        all_pts.append(sweep_pts)
        tag = '  ← anchor' if rel_idx == 0 else ''
        print(f'  [{rel_idx:+d}]  {len(sweep_pts):6,} pts  dt={dt_ms:+7.1f} ms  {fname}{tag}')
    pts_ego = np.concatenate(all_pts, axis=0)
    print(f'  Total: {len(pts_ego):,} pts')
else:
    pts_ego, fname = _dc_load_sweep(nusc, lidar_token, R_e2g_anchor, t_e2g_anchor)
    sd_info = nusc.get('sample_data', lidar_token)
    dt_ms   = (sd_info['timestamp'] - cam_ts) / 1_000.0
    print(f'Single sweep: {len(pts_ego):,} pts  dt={dt_ms:+.1f} ms  ({fname})')

In [ ]:
# Ego-body filter was applied per-sweep in dc_load_lidar (sweep's own ego frame).
print(f'Ego-body filter: applied per-sweep ({"ON" if USE_EGO_BODY_FILTER else "OFF"})')
pts_ego_filt = pts_ego   # alias — ego-body already removed

# Max BEV range filter
_bev_range   = np.sqrt(pts_ego_filt[:, 0]**2 + pts_ego_filt[:, 1]**2)
_n_pre_range = len(pts_ego_filt)
pts_ego_filt = pts_ego_filt[_bev_range <= MAX_LIDAR_RANGE_M]
print(f'Max range filter ({MAX_LIDAR_RANGE_M} m): removed {_n_pre_range - len(pts_ego_filt):,} pts '
      f'→ {len(pts_ego_filt):,} remaining')

# ── Project ego-frame LiDAR into camera ───────────────────────────────────────
fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
pts_cam  = (R_c2e.T @ (pts_ego_filt[:, :3] - t_c2e).T).T   # (N,3) camera frame

in_front = pts_cam[:, 2] > 0.1
u_all    = pts_cam[in_front, 0] / pts_cam[in_front, 2] * fx + cx
v_all    = pts_cam[in_front, 1] / pts_cam[in_front, 2] * fy + cy

u_int_all = np.round(u_all).astype(int)
v_int_all = np.round(v_all).astype(int)
in_img    = (u_int_all >= 0) & (u_int_all < W) & (v_int_all >= 0) & (v_int_all < H)

idx_front   = np.where(in_front)[0]
idx_vis     = idx_front[in_img]           # indices into pts_ego_filt → visible subset (K)

u_int       = np.clip(u_int_all[in_img], 0, W - 1)
v_int       = np.clip(v_int_all[in_img], 0, H - 1)
z_vis       = pts_cam[in_front, 2][in_img]

# pts_ego_vis[i] ←→ u_int[i], v_int[i], z_vis[i]  (same point, different reps)
pts_ego_vis = pts_ego_filt[idx_vis, :3]   # (K, 3)

print(f'\nVisible in {CAMERA}: {len(u_int):,} pts  '
      f'[aggregated ({N_BEFORE}+1+{N_AFTER} sweeps)]')
print(f'Depth range: {z_vis.min():.1f} – {z_vis.max():.1f} m')

# ── Build sparse depth map — min depth per pixel ───────────────────────────────
sparse_depth = np.zeros((H, W), dtype=np.float32)
_tmp = np.full((H, W), np.inf, dtype=np.float32)
np.minimum.at(_tmp, (v_int, u_int), z_vis.astype(np.float32))
sparse_depth[_tmp < np.inf] = _tmp[_tmp < np.inf]

n_pixels = (sparse_depth > 0).sum()
coverage = 100 * n_pixels / (H * W)
print(f'Sparse depth: {n_pixels:,} valid pixels / {H*W:,} total = {coverage:.2f}% coverage')

# Visualise sparse depth
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(img_rgb); axes[0].axis('off'); axes[0].set_title('RGB')
d_vis = np.where(sparse_depth > 0, sparse_depth, np.nan)
im = axes[1].imshow(d_vis, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im, ax=axes[1], label='depth (m)')
axes[1].axis('off'); axes[1].set_title(f'Sparse LiDAR depth ({coverage:.1f}% coverage)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Load CompletionFormer ─────────────────────────────────────────────────────
CFORMER_SRC = str(CFORMER_ROOT / 'src')
CFORMER_DCN = str(CFORMER_ROOT / 'src' / 'model' / 'deformconv')
for _p in [CFORMER_SRC, CFORMER_DCN]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Build args namespace directly — avoids importing config.py which calls
# parse_args() at module level and picks up Jupyter kernel sys.argv
import argparse
args = argparse.Namespace(
    data_name       = 'KITTIDC',
    prop_time       = 6,
    prop_kernel     = 3,
    preserve_input  = True,    # hard-anchor LiDAR values at anchor pixels (MAE → ~0 at those pixels)
    affinity        = 'TGASS',
    affinity_gamma  = 0.5,
    conf_prop       = True,
    legacy          = False,
    from_scratch    = False,
    max_depth       = 90.0,
    num_sample      = 500,
    lidar_lines     = LIDAR_LINES,
)

from model.completionformer import CompletionFormer

cformer = CompletionFormer(args).cuda()

ckpt = torch.load(str(CFORMER_CKPT), map_location='cuda', weights_only=False)
state = ckpt.get('net', ckpt)  # checkpoint may wrap weights under 'net' key
cformer.load_state_dict(state, strict=True)
cformer.eval()
print(f'CompletionFormer loaded from {CFORMER_CKPT.name}')
print(f'Parameters: {sum(p.numel() for p in cformer.parameters())/1e6:.1f} M')
print(f'preserve_input={args.preserve_input}  (set True to hard-anchor LiDAR values)')


In [ ]:
# ── Prepare inputs ────────────────────────────────────────────────────────────
# RGB: (1, 3, H, W)  float32  0-1
rgb_t = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
rgb_t = rgb_t.permute(2, 0, 1).unsqueeze(0).cuda()  # (1,3,H,W)

# Sparse depth: (1, 1, H, W)  float32  metres
dep_t = torch.from_numpy(sparse_depth).unsqueeze(0).unsqueeze(0).cuda()  # (1,1,H,W)

# ── Run CompletionFormer ──────────────────────────────────────────────────────
with torch.no_grad():
    output = cformer({'rgb': rgb_t, 'dep': dep_t})

dense_depth = output['pred'].squeeze().cpu().numpy()  # (H, W)  metres
print(f'Dense depth: shape={dense_depth.shape}  range={dense_depth.min():.2f}–{dense_depth.max():.2f} m')
print(f'Mean depth: {dense_depth[dense_depth>0].mean():.2f} m')


In [ ]:
# ── Visualise: RGB | sparse | completed | error (at LiDAR pixels) ────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 9))

axes[0,0].imshow(img_rgb); axes[0,0].axis('off'); axes[0,0].set_title('RGB')

d_sparse_vis = np.where(sparse_depth > 0, sparse_depth, np.nan)
im1 = axes[0,1].imshow(d_sparse_vis, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im1, ax=axes[0,1], label='m')
axes[0,1].axis('off'); axes[0,1].set_title(f'Sparse LiDAR input ({coverage:.1f}% coverage)')

im2 = axes[1,0].imshow(dense_depth, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im2, ax=axes[1,0], label='m')
axes[1,0].axis('off'); axes[1,0].set_title('CompletionFormer output (dense metric depth)')

# Error at LiDAR anchor pixels
mask  = sparse_depth > 0
error = np.abs(dense_depth - sparse_depth)
err_vis = np.where(mask, error, np.nan)
im3 = axes[1,1].imshow(err_vis, cmap='hot', vmin=0, vmax=1.0)
plt.colorbar(im3, ax=axes[1,1], label='|pred - lidar| (m)')
axes[1,1].axis('off')
mae = float(error[mask].mean())
axes[1,1].set_title(f'Abs error at LiDAR pixels (MAE={mae:.3f} m)')

plt.suptitle(f'Scene {SCENE_IDX}, frame {FRAME_IDX} — {CAMERA}  |  N_BEFORE={N_BEFORE} N_AFTER={N_AFTER}', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Back-project dense depth → (H,W,3) pointmap in PyTorch3D convention ──────
# PyTorch3D: (-X_cam, -Y_cam, Z_cam)
v_grid, u_grid = np.mgrid[0:H, 0:W]
Z = dense_depth.astype(np.float64)
X =  (u_grid - cx) / fx * Z
Y =  (v_grid - cy) / fy * Z

# PyTorch3D convention: negate X and Y
ptmap = np.stack([-X, -Y, Z], axis=-1).astype(np.float32)  # (H, W, 3)
print(f'Pointmap shape: {ptmap.shape}')
print(f'X range: {(-X).min():.2f}–{(-X).max():.2f}')
print(f'Y range: {(-Y).min():.2f}–{(-Y).max():.2f}')
print(f'Z range: {Z.min():.2f}–{Z.max():.2f}')
print('Pointmap ready for SAM3D Objects inference.')


In [ ]:
# ── 3D point cloud: completed depth reprojected to ego frame vs raw LiDAR ────
import plotly.graph_objects as go

Z_f  = dense_depth.astype(np.float32).ravel()
valid_px = Z_f > 0.1
u_all = (u_grid.ravel()[valid_px] - cx) / fx
v_all = (v_grid.ravel()[valid_px] - cy) / fy
z_all = Z_f[valid_px]
pts_cam_dense = np.stack([u_all * z_all, v_all * z_all, z_all], axis=1)

# Camera → ego
pts_ego_dense = (R_c2e @ pts_cam_dense.T).T + t_c2e  # (N,3) ego

# Subsample for plotting
rng   = np.random.default_rng(0)
idx_d = rng.choice(len(pts_ego_dense), min(40_000, len(pts_ego_dense)), replace=False)
idx_l = rng.choice(len(pts_ego_filt),  min(15_000, len(pts_ego_filt)),  replace=False)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_ego_dense[idx_d, 0], y=pts_ego_dense[idx_d, 1], z=pts_ego_dense[idx_d, 2],
    mode='markers',
    marker=dict(size=1.5, color=pts_ego_dense[idx_d, 2], colorscale='turbo',
                cmin=-2, cmax=3, colorbar=dict(title='Z (m)'), showscale=True),
    name='completed depth',
    hovertemplate='X=%{x:.1f} Y=%{y:.1f} Z=%{z:.2f}<extra>completed</extra>',
))
fig.add_trace(go.Scatter3d(
    x=pts_ego_filt[idx_l, 0], y=pts_ego_filt[idx_l, 1], z=pts_ego_filt[idx_l, 2],
    mode='markers',
    marker=dict(size=2, color='white', opacity=0.6),
    name='LiDAR (filtered)',
    hovertemplate='X=%{x:.1f} Y=%{y:.1f} Z=%{z:.2f}<extra>LiDAR</extra>',
))
fig.update_layout(
    title=f'3D point cloud — completed depth vs LiDAR  |  scene {SCENE_IDX} frame {FRAME_IDX}',
    scene=dict(
        xaxis=dict(title='X ego (forward, m)', range=[-5, 60]),
        yaxis=dict(title='Y ego (left, m)',    range=[-30, 30]),
        zaxis=dict(title='Z ego (up, m)',      range=[-3, 5]),
        bgcolor='#0f0f19',
        aspectmode='manual',
        aspectratio=dict(x=3, y=2, z=0.3),
    ),
    paper_bgcolor='#0f0f19',
    font=dict(color='white'),
    legend=dict(bgcolor='#222'),
    height=600,
    margin=dict(l=0,r=0,t=40,b=0),
)
fig.show()

---

## 7. Convex Hull Depth Anchoring — Per-Object Prototype

Tests whether clamping the CFormer dense depth inside the **convex hull** of a SAM3 mask fixes the occluder-contamination problem (pole/sign in front of a car pulling the mesh forward).

**SAM3 mask structure**: SAM3 produces **one mask per object**, not two. When a pole is in front of a car the binary mask has a concavity/notch at the occluder location — it is one connected region with a non-convex outline. The convex hull fills that notch, covering the full spatial extent of the object in image space.

**Why anchoring helps**: CFormer is RGB-guided. Even with no LiDAR anchor at the pole pixels (HDBSCAN already removed pole returns from the sparse map), CFormer sees the pole texture and predicts shallow depth there. With `preserve_input=True` it won't override those unanchored pixels from LiDAR — but it will use RGB cues. The hull clamp is a post-CFormer fix: any pixel inside the car's spatial footprint that CFormer predicted as shallower than the car's front surface gets raised to `z_floor`.

**Per-object design note**: in the full pipeline this would be applied per object, each with its own modified pointmap. Clamping mask #2's hull does not corrupt a closer object whose mask overlaps the same spatial region — because that closer object would be processed separately with its own (lower) z_floor.

**Workflow**:
1. Config cell — set `SAM3_PROMPT`, `MASK_IDX`, `HULL_ANCHORING`, `HULL_Z_MARGIN`
2. SAM3 cell — run inference, view all instances, pick `MASK_IDX`
3. Convex hull cell — visualise mask notch and hull fill
4. HDBSCAN cell — isolate dominant surface cluster, compute `z_floor`
5. CFormer cell — build O5-cleaned sparse anchor, run CFormer, apply hull clamp
6. Comparison cell — side-by-side and histogram: raw vs anchored in-hull depth

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Config — convex hull depth anchoring prototype
# ══════════════════════════════════════════════════════════════════════════════
SAM3_PROMPT    = 'car'    # text prompt for SAM3 (must match a pipeline class)
MASK_IDX       = 2        # which detected instance to analyse — update after viewing overlay
HULL_ANCHORING = True     # False = raw CFormer only; True = apply hull depth clamp
HULL_Z_MARGIN  = 1.0      # m: z_floor = 5th-pct cluster depth − this margin

from huggingface_hub import hf_hub_download
SAM3_CKPT = Path(hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt'))

# HDBSCAN params — per class, identical to full pipeline
HDBSCAN_CFG = {
    'car':                 dict(min_cluster_size=5, min_samples=1, cluster_eps=0.50),
    'pedestrian':          dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'bicycle and cyclist': dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'motorcycle':          dict(min_cluster_size=3, min_samples=1, cluster_eps=0.30),
    'truck':               dict(min_cluster_size=5, min_samples=1, cluster_eps=0.60),
    'bus':                 dict(min_cluster_size=5, min_samples=1, cluster_eps=0.80),
}
hdbscan_params = HDBSCAN_CFG.get(
    SAM3_PROMPT, dict(min_cluster_size=3, min_samples=1, cluster_eps=0.40))

print(f'SAM3_PROMPT    = {SAM3_PROMPT!r}')
print(f'MASK_IDX       = {MASK_IDX}  ← update after viewing the mask overlay below')
print(f'HULL_ANCHORING = {HULL_ANCHORING}')
print(f'HULL_Z_MARGIN  = {HULL_Z_MARGIN} m')
print(f'HDBSCAN params : {hdbscan_params}')
print(f'SAM3_CKPT      : {SAM3_CKPT}')


In [ ]:
# ── Load SAM3 → run inference → unload → reload CFormer ──────────────────────
# SAM3 needs most GPU memory; CFormer is offloaded temporarily.
import gc, tempfile, shutil
from PIL import Image as _PIL_Image

_cformer_sd = cformer.state_dict()
del cformer
torch.cuda.empty_cache(); gc.collect()
print(f'CFormer offloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

_SAM3_SRC = str(DEV_ROOT / 'models' / 'SAM3')
if _SAM3_SRC not in sys.path:
    sys.path.insert(0, _SAM3_SRC)
from sam3.model_builder import build_sam3_video_predictor

_sam3 = build_sam3_video_predictor(
    gpus_to_use=range(torch.cuda.device_count()),
    checkpoint_path=str(SAM3_CKPT),
)
print('SAM3 loaded.')

def _run_sam3_frame(predictor, img_rgb, text_prompt, score_thresh=0.5, min_mask_px=100):
    tmp = tempfile.mkdtemp(prefix='sam3_hull_')
    try:
        _PIL_Image.fromarray(img_rgb).save(os.path.join(tmp, '00000.jpg'))
        sid = predictor.handle_request(
            dict(type='start_session', resource_path=tmp))['session_id']
        predictor.handle_request(
            dict(type='add_prompt', session_id=sid, frame_index=0, text=text_prompt))
        frame_outs = {}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            for r in predictor.handle_stream_request(
                    dict(type='propagate_in_video', session_id=sid)):
                frame_outs[r['frame_index']] = r['outputs']
        predictor.handle_request(dict(type='close_session', session_id=sid))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    out = frame_outs.get(0, {})
    dets = []
    for oid, prob, msk, box in zip(out.get('out_obj_ids', []),
                                    out.get('out_probs', []),
                                    out.get('out_binary_masks', []),
                                    out.get('out_boxes_xywh', [])):
        if prob < score_thresh:
            continue
        bm = np.array(msk).squeeze().astype(bool)
        if bm.sum() < min_mask_px:
            continue
        dets.append(dict(binary_mask=bm, score=float(prob),
                         box_xywh=np.array(box), prompt=text_prompt))
    return dets

with torch.no_grad():
    sam3_dets = _run_sam3_frame(_sam3, img_rgb, SAM3_PROMPT)
print(f'{len(sam3_dets)} detections  (prompt={SAM3_PROMPT!r})')

# Unload SAM3, reload CFormer
del _sam3
torch.cuda.empty_cache(); gc.collect()

# SAM3 leaves torch.autocast(BF16) globally enabled — disable it before CFormer
# (same fix as autolabeling_pipeline.ipynb §5; DCNv2 kernel breaks under BF16 autocast)
print(f'Autocast state after SAM3: {torch.is_autocast_enabled()}')
torch.autocast('cuda', enabled=False).__enter__()
print(f'Autocast disabled: {not torch.is_autocast_enabled()}')

cformer = CompletionFormer(args).cuda()
cformer.load_state_dict(_cformer_sd); cformer.eval()
print(f'CFormer reloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

# ── Show all masks with index labels → set MASK_IDX in config cell ────────────
overlay = img_rgb.copy().astype(np.float32)
for det in sam3_dets:
    overlay[det['binary_mask']] = (overlay[det['binary_mask']] * 0.45
                                    + np.array([60, 180, 60], np.float32) * 0.55)
fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(overlay.astype(np.uint8))
for idx, det in enumerate(sam3_dets):
    ys, xs = np.where(det['binary_mask'])
    ax.text(xs.mean(), ys.mean(), str(idx), color='white', fontsize=9,
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
ax.axis('off')
ax.set_title(f'SAM3 — {len(sam3_dets)} instances  (prompt={SAM3_PROMPT!r})\n'
             f'↑ Set MASK_IDX in the config cell to the index you want to analyse, '
             f'then run the cells below')
plt.tight_layout(); plt.show()


In [ ]:
# ── Select mask, build convex hull, visualise ─────────────────────────────────
# SAM3 produces ONE mask per object — a single connected region with a concavity
# where the occluder (pole/sign) is.  The convex hull fills that notch.

assert 0 <= MASK_IDX < len(sam3_dets), \
    f'MASK_IDX={MASK_IDX} out of range — {len(sam3_dets)} detections (0–{len(sam3_dets)-1})'

binary_mask = sam3_dets[MASK_IDX]['binary_mask']   # (H, W) bool

# Convex hull via OpenCV contour (handles notched / non-convex masks cleanly)
mask_u8     = binary_mask.astype(np.uint8) * 255
contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
all_pts     = np.vstack([c.reshape(-1, 2) for c in contours])   # (M, 2)  x,y
hull_pts    = cv2.convexHull(all_pts)                            # (K, 1, 2)

hull_filled_u8 = np.zeros((H, W), dtype=np.uint8)
cv2.fillPoly(hull_filled_u8, [hull_pts], 255)
hull_mask   = hull_filled_u8 > 0
gap_mask    = hull_mask & ~binary_mask   # notch fill = hull minus SAM3 mask

hull_outline = np.zeros((H, W), dtype=np.uint8)
cv2.drawContours(hull_outline, [hull_pts], -1, 255, 2)

print(f'SAM3 mask pixels   : {binary_mask.sum():,}')
print(f'Convex hull pixels : {hull_mask.sum():,}')
print(f'Notch fill (gap)   : {gap_mask.sum():,}  '
      f'({100*gap_mask.sum()/max(hull_mask.sum(),1):.1f}% of hull — occluder region)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_rgb); axes[0].axis('off')
axes[0].set_title(f'RGB — mask {MASK_IDX} ({SAM3_PROMPT})')

vis0 = img_rgb.copy().astype(np.float32)
vis0[binary_mask] = vis0[binary_mask] * 0.4 + np.array([60, 180, 60]) * 0.6
axes[1].imshow(vis0.astype(np.uint8)); axes[1].axis('off')
axes[1].set_title('SAM3 binary mask (green)\n— concavity/notch = occluder region')

vis1 = img_rgb.copy().astype(np.float32)
vis1[binary_mask]       = vis1[binary_mask]       * 0.4 + np.array([60, 180, 60])  * 0.6
vis1[gap_mask]          = vis1[gap_mask]           * 0.4 + np.array([255, 140, 0])  * 0.6
vis1[hull_outline > 0]  = [255, 255, 0]
axes[2].imshow(vis1.astype(np.uint8)); axes[2].axis('off')
axes[2].set_title('Convex hull:\ngreen=SAM3 mask  orange=notch fill  yellow=hull outline')

plt.suptitle(f'Mask {MASK_IDX} — convex hull (fills occluder gap)', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── In-mask LiDAR overlay — sanity check (mirrors lidar_filtering_exploration §5) ──
# Blue  = SAM3 binary mask
# Orange dots = filtered LiDAR points that project inside the mask
# Compare this point count with lidar_filtering_exploration to confirm the
# ego-body filter and index chain produce identical in-mask results.

_u = np.clip(u_int, 0, W - 1)
_v = np.clip(v_int, 0, H - 1)
_in_mask = binary_mask[_v, _u]
_idx_in  = np.where(_in_mask)[0]

u_inmask_viz = u_int[_idx_in].astype(float)
v_inmask_viz = v_int[_idx_in].astype(float)
Z_inmask_viz = z_vis[_idx_in]

print(f'In-mask LiDAR points: {len(_idx_in)} / {len(u_int)} visible '
      f'(depth {Z_inmask_viz.min():.1f}–{Z_inmask_viz.max():.1f} m)'
      if len(_idx_in) > 0 else f'In-mask LiDAR points: 0 / {len(u_int)} visible')

vis_mask = img_rgb.copy().astype(np.float32)
vis_mask[binary_mask] = vis_mask[binary_mask] * 0.4 + np.array([100, 200, 255]) * 0.6

for i in range(len(u_inmask_viz)):
    cv2.circle(vis_mask, (int(u_inmask_viz[i]), int(v_inmask_viz[i])), 3, (255, 120, 0), -1)

plt.figure(figsize=(14, 5))
plt.imshow(vis_mask.astype(np.uint8))
plt.axis('off')
plt.title(f'Mask "{SAM3_PROMPT}" #{MASK_IDX} (blue) | {len(_idx_in)} in-mask LiDAR points (orange)')
plt.tight_layout()
plt.show()

In [ ]:
# ── HDBSCAN: two-pass clustering → dominant in-mask surface → min_depth ───────
# Mirrors lidar_filtering_exploration.ipynb §6 exactly.
#
# Inputs from dc_project : u_int, v_int, z_vis, pts_ego_vis  (K visible pts)
# Inputs from a56c6604  : binary_mask, hull_mask, hull_outline
# Inputs from 6555381a  : hdbscan_params, HULL_Z_MARGIN
import hdbscan as _hdbscan

in_mask_bool = binary_mask[v_int, u_int]     # (K,) bool — which visible pts fall in mask
idx_in       = np.where(in_mask_bool)[0]     # indices into u_int / v_int / z_vis / pts_ego_vis
print(f'In-mask LiDAR points : {len(idx_in)} / {len(u_int)} visible')

# ── Initialise outputs ────────────────────────────────────────────────────────
min_depth           = None
dominant_idx_global = np.array([], dtype=int)
labels_raw          = np.full(len(idx_in), -1, dtype=int)
labels_merged       = np.full(len(idx_in), -1, dtype=int)
n_clusters_raw      = 0
n_clusters          = 0
_dom_c              = -1

if len(idx_in) >= hdbscan_params['min_cluster_size']:
    pts_in_ego = pts_ego_vis[idx_in]   # (n_in, 3) ego-frame XYZ of in-mask points

    # Pass 1 — raw subclusters (no epsilon) — diagnostic only
    _c_raw = _hdbscan.HDBSCAN(
        min_cluster_size = hdbscan_params['min_cluster_size'],
        min_samples      = hdbscan_params['min_samples'],
        metric           = 'euclidean',
    )
    labels_raw     = _c_raw.fit_predict(pts_in_ego)
    n_clusters_raw = int(labels_raw.max()) + 1 if labels_raw.max() >= 0 else 0

    # Pass 2 — merged clusters (with cluster_selection_epsilon)
    _c_merged = _hdbscan.HDBSCAN(
        min_cluster_size          = hdbscan_params['min_cluster_size'],
        min_samples               = hdbscan_params['min_samples'],
        metric                    = 'euclidean',
        cluster_selection_epsilon = hdbscan_params['cluster_eps'],
    )
    labels_merged = _c_merged.fit_predict(pts_in_ego)
    n_clusters    = int(labels_merged.max()) + 1 if labels_merged.max() >= 0 else 0

    print(f'Without ε         : {n_clusters_raw} cluster(s), {(labels_raw == -1).sum()} noise pts')
    print(f'With ε={hdbscan_params["cluster_eps"]}m : {n_clusters} cluster(s), {(labels_merged == -1).sum()} noise pts')

    if n_clusters > 0:
        _counts  = np.array([(labels_merged == c).sum() for c in range(n_clusters)])
        _order   = np.argsort(_counts)[::-1]   # descending by point count
        _biggest = int(_counts[_order[0]])

        # Proximity selection: if two similarly-sized clusters exist pick the
        # closer one — rejects background walls with similar in-mask point counts.
        PROXIMITY_MIN_PTS = 30
        PROXIMITY_RATIO   = 0.70
        if (_biggest >= PROXIMITY_MIN_PTS
                and len(_order) >= 2
                and _counts[_order[1]] >= PROXIMITY_RATIO * _biggest):
            _c0 = int(_order[0]); _c1 = int(_order[1])
            _d0 = float(np.linalg.norm(pts_in_ego[labels_merged == _c0].mean(axis=0)))
            _d1 = float(np.linalg.norm(pts_in_ego[labels_merged == _c1].mean(axis=0)))
            _dom_c = _c0 if _d0 <= _d1 else _c1
            _mode  = f'proximity (#{_c0}@{_d0:.1f}m vs #{_c1}@{_d1:.1f}m)'
        else:
            _dom_c = int(_order[0])
            _mode  = 'biggest'

        print(f'\nCluster selection ({_mode}):')
        for c in range(n_clusters):
            tag = '  ← SELECTED' if c == _dom_c else ''
            print(f'  Cluster {c}: {_counts[c]} pts{tag}')

        is_dominant         = labels_merged == _dom_c
        dominant_idx_global = idx_in[is_dominant]       # indices back into u_int / z_vis
        z_dominant          = z_vis[dominant_idx_global]  # camera depth [m] of dominant pts

        min_depth = float(np.percentile(z_dominant, 5)) - HULL_Z_MARGIN
        print(f'\nDominant cluster   : {is_dominant.sum()} pts  '
              f'cam-depth {z_dominant.min():.2f}–{z_dominant.max():.2f} m')
        print(f'min_depth          : 5th-pct ({np.percentile(z_dominant, 5):.2f} m) '
              f'− margin ({HULL_Z_MARGIN} m) = {min_depth:.2f} m')
        print(f'→ CFormer not allowed to predict closer than {min_depth:.2f} m inside hull')
    else:
        print('HDBSCAN found no clusters — hull anchoring will be skipped')
else:
    print(f'Too few in-mask LiDAR points ({len(idx_in)}) — hull anchoring skipped')

# ── Visualise ─────────────────────────────────────────────────────────────────
CLUSTER_PALETTE = [
    (255, 80, 80), (80, 220, 80), (80, 120, 255), (255, 200, 0),
    (200, 80, 255), (0, 210, 210), (255, 100, 0), (60, 255, 200),
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img_rgb, alpha=0.65)

if len(idx_in) > 0 and n_clusters > 0:
    noise_in = idx_in[labels_merged == -1]
    if len(noise_in) > 0:
        ax.scatter(u_int[noise_in], v_int[noise_in], s=8, c='red', alpha=0.7,
                   label=f'noise ({len(noise_in)} pts)')
    for c in range(n_clusters):
        c_global = idx_in[labels_merged == c]
        col = np.array(CLUSTER_PALETTE[c % len(CLUSTER_PALETTE)]) / 255
        tag = ' [SELECTED]' if c == _dom_c else ''
        sz  = 12 if c == _dom_c else 6
        ax.scatter(u_int[c_global], v_int[c_global], s=sz, color=col, alpha=0.9,
                   label=f'cluster {c} ({len(c_global)} pts){tag}')
elif len(idx_in) > 0:
    ax.scatter(u_int[idx_in], v_int[idx_in], s=8, c='red', alpha=0.7,
               label=f'all noise ({len(idx_in)} pts)')

_hy, _hx = np.where(hull_outline > 0)
ax.scatter(_hx, _hy, s=1, c='yellow', alpha=0.3, label='hull boundary')

ttl = (f'Mask {MASK_IDX} ({SAM3_PROMPT}) — HDBSCAN (ε={hdbscan_params["cluster_eps"]}m)  '
       f'|  min_depth = {min_depth:.2f} m'
       if min_depth is not None
       else f'Mask {MASK_IDX} ({SAM3_PROMPT}) — HDBSCAN  (no cluster — anchoring skipped)')
ax.set_title(ttl)
ax.legend(loc='upper right', fontsize=8)
ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── Build O5-cleaned sparse map → CFormer → hull anchoring ───────────────────
keep = np.ones(len(u_int), dtype=bool)
if len(dominant_idx_global) > 0:
    keep[idx_in]              = False
    keep[dominant_idx_global] = True
    print(f'Sparse anchor: removed {idx_in.size - dominant_idx_global.size} '
          f'non-dominant in-mask pts')
else:
    print('No dominant cluster — using all in-mask LiDAR unchanged')

sparse_clean = np.zeros((H, W), dtype=np.float32)
_tmp2 = np.full((H, W), np.inf, dtype=np.float32)
np.minimum.at(_tmp2, (v_int[keep], u_int[keep]), z_vis[keep].astype(np.float32))
sparse_clean[_tmp2 < np.inf] = _tmp2[_tmp2 < np.inf]
print(f'Cleaned sparse: {(sparse_clean > 0).sum():,} pixels  '
      f'(full global had {(sparse_depth > 0).sum():,})')

dep_clean_t = torch.from_numpy(sparse_clean).unsqueeze(0).unsqueeze(0).cuda()
with torch.no_grad():
    _out = cformer({'rgb': rgb_t, 'dep': dep_clean_t})
dense_raw = _out['pred'].squeeze().cpu().numpy()
print(f'CFormer output range: {dense_raw.min():.2f}–{dense_raw.max():.2f} m')

# ── Hull anchoring: clamp to >= min_depth (minimum distance from camera) ──────
# min_depth = closest distance CFormer is allowed to predict inside the hull.
# Anything shallower = occluder contamination; clamp it to the object's depth.
dense_anchored = dense_raw.copy()
n_clamped = 0
if min_depth is not None:
    _before = dense_anchored[hull_mask].copy()
    dense_anchored[hull_mask] = np.maximum(dense_anchored[hull_mask], min_depth)
    n_clamped = int((_before < min_depth).sum())
    print(f'Hull anchoring: clamped {n_clamped} / {hull_mask.sum()} hull pixels '
          f'(predicted closer than {min_depth:.2f} m)')
else:
    print('Hull anchoring skipped — no dominant cluster')

dense_out    = dense_anchored if HULL_ANCHORING else dense_raw
active_label = ('hull-anchored' if (HULL_ANCHORING and min_depth is not None)
                else 'CFormer raw')
print(f'\nActive output: {active_label}')

# ── Full-frame side-by-side ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img_rgb); axes[0].axis('off'); axes[0].set_title('RGB')

im1 = axes[1].imshow(dense_raw, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im1, ax=axes[1], label='m'); axes[1].axis('off')
axes[1].set_title('CFormer — O5-cleaned input (raw output)')

_anc_show = dense_anchored if min_depth is not None else dense_raw
im2 = axes[2].imshow(_anc_show, cmap='turbo', vmin=0, vmax=60)
plt.colorbar(im2, ax=axes[2], label='m'); axes[2].axis('off')
axes[2].set_title(
    f'Hull-anchored (min_depth={min_depth:.2f} m, {n_clamped} px clamped)'
    if min_depth is not None else 'Hull anchoring skipped (no cluster)')

for ax in axes[1:]:
    _hy, _hx = np.where(hull_outline > 0)
    ax.scatter(_hx, _hy, s=1, c='yellow', alpha=0.3)

plt.suptitle(f'CFormer + hull anchoring  |  mask {MASK_IDX} ({SAM3_PROMPT})  '
             f'HULL_ANCHORING={HULL_ANCHORING}', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── In-mask depth comparison: raw CFormer vs hull-anchored ───────────────────
rows_h = np.where(hull_mask.any(axis=1))[0]
cols_h = np.where(hull_mask.any(axis=0))[0]
r0, r1 = rows_h[0], rows_h[-1]
c0, c1 = cols_h[0], cols_h[-1]
pad = 40
r0c = max(0, r0 - pad); r1c = min(H, r1 + pad)
c0c = max(0, c0 - pad); c1c = min(W, c1 + pad)

depth_raw_inmask = np.where(hull_mask, dense_raw,      np.nan)
depth_anc_inmask = np.where(hull_mask, dense_anchored, np.nan)
vmin = float(np.nanmin(depth_raw_inmask)) * 0.9
vmax = float(np.nanmax(depth_raw_inmask)) * 1.1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_rgb[r0c:r1c, c0c:c1c])
_hy, _hx = np.where(hull_outline[r0c:r1c, c0c:c1c] > 0)
axes[0].scatter(_hx, _hy, s=1, c='yellow', alpha=0.5)
axes[0].axis('off'); axes[0].set_title(f'RGB crop — mask {MASK_IDX} ({SAM3_PROMPT})')

im1 = axes[1].imshow(depth_raw_inmask[r0c:r1c, c0c:c1c], cmap='turbo', vmin=vmin, vmax=vmax)
plt.colorbar(im1, ax=axes[1], label='depth (m)')
axes[1].axis('off')
axes[1].set_title('CFormer raw output\n[shallow region = occluder contamination]')

im2 = axes[2].imshow(depth_anc_inmask[r0c:r1c, c0c:c1c], cmap='turbo', vmin=vmin, vmax=vmax)
plt.colorbar(im2, ax=axes[2], label='depth (m)')
axes[2].axis('off')
_md_str = f'min_depth={min_depth:.2f} m' if min_depth is not None else 'no min_depth'
_active = ' ← ACTIVE' if HULL_ANCHORING else ''
axes[2].set_title(f'Hull-anchored ({_md_str}){_active}\n'
                  f'[occluder clamped: not closer than {_md_str}]')

plt.suptitle(f'In-hull depth — mask {MASK_IDX} ({SAM3_PROMPT})  |  '
             f'HULL_ANCHORING={HULL_ANCHORING}', y=1.01)
plt.tight_layout(); plt.show()

# Histogram
fig, ax = plt.subplots(figsize=(10, 3))
bins = np.linspace(vmin, vmax, 60)
ax.hist(depth_raw_inmask[hull_mask], bins=bins, alpha=0.55,
        color='steelblue', label='CFormer raw')
ax.hist(depth_anc_inmask[hull_mask], bins=bins, alpha=0.55,
        color='tomato',   label='Hull-anchored')
if min_depth is not None:
    ax.axvline(min_depth, color='gold', lw=2, ls='--',
               label=f'min_depth = {min_depth:.2f} m')
ax.set_xlabel('camera depth (m)'); ax.set_ylabel('pixel count')
ax.set_title('In-hull depth histogram — shallow peak left of min_depth = occluder contamination')
ax.legend(); plt.tight_layout(); plt.show()
